# Bivariate Global Maps of Front-Associated Properties

This notebook loads the co-located output from `build_v3` and creates
bivariate global maps where each front is colored by **two properties**
simultaneously: **divergence/|f|** (horizontal divergence normalized by
the absolute Coriolis parameter) and **Rossby number**.

**Sections:**
1. Configuration
2. Data Loading & Inspection
3. Derived Variables
4. Bivariate Color Mapping Functions
5. Plotting Functions
6. Generate Bivariate Map

---
## 1. Configuration

Edit this cell to change input paths, variables, and plotting options.

In [ ]:
import os
from pathlib import Path

# ─── DATA PATHS ───────────────────────────────────────────────────────────────
# Root of the OGCM data tree (must contain LLC/Fronts/...)
OGCM_ROOT = os.environ.get('OS_OGCM', '/path/to/OGCM')

# build_v3 co-located output directory
# Standard layout: $OS_OGCM/LLC/Fronts/V3/<YYYYMMDD_HHMMSS>/
RESULTS_DIR = Path(OGCM_ROOT) / 'LLC' / 'Fronts' / 'group_fronts' / 'v3'

# Timestamp of the snapshot (ISO 8601 format for viz_loaders)
TIME_STR = '2012-11-09T12:00:00'

# Run tag (suffix on output filenames)
RUN_TAG = 'v3_bin_D'

# LLC coordinate file (None → default $OS_OGCM/LLC/Fronts/coords/LLC_coords_lat_lon.nc)
COORDS_FILE = None

# ─── VARIABLE SELECTION ───────────────────────────────────────────────────────
# Variable 1 is DERIVED: divergence / |coriolis_f|
# We need two base columns from the co-located parquet to compute it.
DIV_BASE   = 'divergence'    # base name for horizontal divergence
CORI_BASE  = 'coriolis_f'    # base name for Coriolis parameter f

# Variable 2 is read directly from the parquet.
ROSSBY_BASE = 'rossby_number'  # base name for Rossby number

# Display labels for axes / titles
VAR1_LABEL = r'$\delta / |f|$'       # divergence / |f|
VAR2_LABEL = r'$Ro$'                  # Rossby number

# Summary statistic to use: 'mean', 'median', or 'p90'
STATISTIC = 'mean'

# ─── BIVARIATE BINNING ────────────────────────────────────────────────────────
# Number of bins per axis (N×N color grid). Start with 3 or 4.
N_BINS = 3

# Percentile clipping for bin edges (removes outlier tails)
CLIP_PERCENTILE = 2  # clip bottom/top N%

# ─── MAP OPTIONS ──────────────────────────────────────────────────────────────
# Projection: 'Robinson', 'Mollweide', 'PlateCarree'
PROJECTION = 'Robinson'

# Marker size for front scatter points
MARKER_SIZE = 1.0

# Spatial binning: if True, bin fronts into 2° cells and plot the
# dominant bivariate category per cell. If False, scatter each front centroid.
USE_SPATIAL_BINNING = True
SPATIAL_BIN_DEG = 2  # bin size in degrees (only used if USE_SPATIAL_BINNING=True)

# ─── OUTPUT ────────────────────────────────────────────────────────────────────
SAVE_DIR = Path('.') / 'figures'
SAVE_DPI = 200

---
## 2. Data Loading & Inspection

In [ ]:
import sys
import numpy as np
import pandas as pd

# Ensure the fronts package is importable
from fronts.properties.viz_loaders import (
    load_global_front_results,
    load_colocation_table,
    load_geometry_table,
    merge_geometry_colocation,
)

In [ ]:
# Load all front results (geometry + colocation merged, coords aligned)
results = load_global_front_results(
    results_dir=RESULTS_DIR,
    time_str=TIME_STR,
    run_tag=RUN_TAG,
    coords_file=COORDS_FILE,
)

df_enriched    = results['df_enriched']
lat_global     = results['lat_global']
lon_global     = results['lon_global']
labeled_global = results['labeled_global']
metadata       = results['metadata']

print(f"Enriched DataFrame: {df_enriched.shape[0]:,} fronts, {df_enriched.shape[1]} columns")
print(f"Grid shape: {lat_global.shape}")

In [ ]:
# ─── INSPECT AVAILABLE COLUMNS ────────────────────────────────────────────────
# Print all columns so we can verify the variable names exist.
print("All columns in df_enriched:")
print("-" * 50)
for col in sorted(df_enriched.columns):
    print(f"  {col}")
print(f"\nTotal: {len(df_enriched.columns)} columns")

In [ ]:
# ─── RESOLVE COLUMN NAMES ─────────────────────────────────────────────────────
# Build the full column name from base variable name + statistic suffix.
# Falls back through suffix priority if the exact name is missing.

def resolve_column(df, base_name, statistic):
    """Find the best matching column for `base_name` with `statistic` suffix.

    Tries: '{base}_{stat}', '{base}_median', '{base}_mean', '{base}'
    Returns the column name or raises KeyError with helpful message.
    """
    candidates = [
        f"{base_name}_{statistic}",
        f"{base_name}_median",
        f"{base_name}_mean",
        base_name,
    ]
    for c in candidates:
        if c in df.columns:
            return c

    # Provide a helpful error listing columns that partially match
    matches = [c for c in df.columns if base_name in c]
    raise KeyError(
        f"Could not find column for '{base_name}' with statistic '{statistic}'.\n"
        f"Tried: {candidates}\n"
        f"Partial matches in DataFrame: {matches}\n"
        f"→ Update DIV_BASE / CORI_BASE / ROSSBY_BASE in the configuration cell."
    )


# Resolve the three base columns we need
div_col   = resolve_column(df_enriched, DIV_BASE,   STATISTIC)
cori_col  = resolve_column(df_enriched, CORI_BASE,  STATISTIC)
ro_col    = resolve_column(df_enriched, ROSSBY_BASE, STATISTIC)

print(f"Divergence : {DIV_BASE!r:25s} →  column '{div_col}'")
print(f"Coriolis f : {CORI_BASE!r:25s} →  column '{cori_col}'")
print(f"Rossby num : {ROSSBY_BASE!r:25s} →  column '{ro_col}'")

# Quick summary stats for the raw columns
for c in [div_col, cori_col, ro_col]:
    vals = df_enriched[c].dropna()
    print(f"\n  {c}: n={len(vals):,}, "
          f"min={vals.min():.3e}, median={vals.median():.3e}, "
          f"max={vals.max():.3e}")

---
## 3. Derived Variables

**Variable 1** is not a raw column — it is the ratio
$\delta / |f|$ (horizontal divergence normalized by the absolute
Coriolis parameter).  We compute it here from the resolved columns.

**Variable 2** (Rossby number) is read directly.

In [ ]:
# ─── COMPUTE DIVERGENCE / |f| ─────────────────────────────────────────────────
# Guard against division by zero: set |f| < tiny to NaN so those fronts
# (very near the equator where f → 0) are excluded from the bivariate map.

div_vals  = df_enriched[div_col].values.astype(np.float64)
cori_vals = df_enriched[cori_col].values.astype(np.float64)

abs_f = np.abs(cori_vals)
# Threshold: |f| below ~0.5° latitude ≈ 1.3e-6 s⁻¹ is unreliable
F_MIN = 1e-6
abs_f_safe = np.where(abs_f > F_MIN, abs_f, np.nan)

div_over_f = div_vals / abs_f_safe

# Store back into the DataFrame for convenience
df_enriched['div_over_absf'] = div_over_f

n_valid = np.isfinite(div_over_f).sum()
n_excluded = (~np.isfinite(div_over_f)).sum() - np.isnan(div_vals).sum()
print(f"div/|f|: {n_valid:,} valid fronts")
print(f"  ({n_excluded:,} excluded due to |f| < {F_MIN:.0e})")
print(f"  range: [{np.nanmin(div_over_f):.3e}, {np.nanmax(div_over_f):.3e}]")
print(f"  median: {np.nanmedian(div_over_f):.3e}")

In [ ]:
# ─── EXTRACT THE TWO FINAL ARRAYS ─────────────────────────────────────────────
# values1 = divergence / |f|  (derived)
# values2 = Rossby number     (direct column)

values1 = df_enriched['div_over_absf'].values
values2 = df_enriched[ro_col].values.astype(np.float64)

# Column names used in plot titles / legends
col1 = 'div_over_absf'
col2 = ro_col

print(f"Variable 1 (x-axis): div/|f|  — {np.isfinite(values1).sum():,} finite values")
print(f"Variable 2 (y-axis): {ro_col} — {np.isfinite(values2).sum():,} finite values")

---
## 4. Bivariate Color Mapping Functions

The bivariate color scheme works as follows:

1. **Bin each variable** into `N` quantile-based bins (e.g., terciles for N=3).
2. **Build an N×N color grid** where one axis represents variable 1 and the other
   variable 2. Each cell in the grid gets a unique color.
3. **Assign each front** a color based on which (bin_var1, bin_var2) cell it falls into.

The color grid uses a perceptually reasonable 2D interpolation:
- Bottom-left corner: low var1, low var2
- Top-right corner: high var1, high var2
- The four corners are anchored to distinct hues, and interior cells
  are linearly interpolated in RGB space.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from scipy.stats import binned_statistic_2d


def make_bivariate_colormap(n_bins, corner_colors=None):
    """Create an N×N bivariate color grid by interpolating four corner colors.

    Parameters
    ----------
    n_bins : int
        Number of bins per axis (produces n_bins × n_bins colors).
    corner_colors : dict, optional
        RGB tuples for the four corners:
          'low_low'   : bottom-left  (low var1, low var2)
          'high_low'  : bottom-right (high var1, low var2)
          'low_high'  : top-left     (low var1, high var2)
          'high_high' : top-right    (high var1, high var2)
        Defaults to a gray–blue–red–purple scheme.

    Returns
    -------
    color_grid : np.ndarray, shape (n_bins, n_bins, 3)
        RGB colors indexed as [var2_bin, var1_bin, rgb].
        var2_bin=0 is the lowest bin (bottom row in the legend).
    """
    if corner_colors is None:
        # Classic bivariate scheme:
        #   low-low  = light gray    high-low  = blue
        #   low-high = red           high-high = dark purple
        corner_colors = {
            'low_low':   np.array([0.87, 0.87, 0.87]),  # light gray
            'high_low':  np.array([0.24, 0.31, 0.71]),  # blue
            'low_high':  np.array([0.83, 0.24, 0.30]),  # red
            'high_high': np.array([0.33, 0.15, 0.46]),  # dark purple
        }

    c = corner_colors
    color_grid = np.zeros((n_bins, n_bins, 3))

    for i in range(n_bins):       # var2 axis (rows: bottom=0=low)
        for j in range(n_bins):   # var1 axis (columns: left=0=low)
            # Fractional position in [0, 1]
            fx = j / max(n_bins - 1, 1)  # var1 fraction
            fy = i / max(n_bins - 1, 1)  # var2 fraction

            # Bilinear interpolation across the four corners
            color_grid[i, j] = (
                (1 - fx) * (1 - fy) * c['low_low']
                +    fx  * (1 - fy) * c['high_low']
                + (1 - fx) *    fy  * c['low_high']
                +    fx  *    fy    * c['high_high']
            )

    return np.clip(color_grid, 0, 1)


def assign_bivariate_bins(values1, values2, n_bins, clip_pct=2):
    """Bin two arrays into N quantile-based bins and return bin indices.

    Parameters
    ----------
    values1, values2 : array-like
        The two property arrays (one value per front).
    n_bins : int
        Number of bins per variable.
    clip_pct : float
        Percentile at which to clip outlier tails (applied symmetrically).

    Returns
    -------
    bins1, bins2 : np.ndarray of int
        Bin index (0-based) for each front. -1 where data is NaN/invalid.
    edges1, edges2 : np.ndarray
        Bin edge arrays (length n_bins+1) for labeling.
    """
    v1 = np.asarray(values1, dtype=np.float64)
    v2 = np.asarray(values2, dtype=np.float64)

    # Only use valid (finite) values to compute bin edges
    valid = np.isfinite(v1) & np.isfinite(v2)

    def _edges(v, valid_mask):
        """Compute clipped quantile-based bin edges."""
        vv = v[valid_mask]
        lo = np.percentile(vv, clip_pct)
        hi = np.percentile(vv, 100 - clip_pct)
        # Quantile edges within the clipped range
        vv_clipped = vv[(vv >= lo) & (vv <= hi)]
        edges = np.quantile(vv_clipped, np.linspace(0, 1, n_bins + 1))
        # Extend edges slightly to capture all data
        edges[0] = lo
        edges[-1] = hi
        return edges

    edges1 = _edges(v1, valid)
    edges2 = _edges(v2, valid)

    # Digitize: np.digitize returns 1-based indices; clip to [0, n_bins-1]
    bins1 = np.digitize(v1, edges1) - 1
    bins2 = np.digitize(v2, edges2) - 1
    bins1 = np.clip(bins1, 0, n_bins - 1)
    bins2 = np.clip(bins2, 0, n_bins - 1)

    # Mark invalid entries
    bins1[~valid] = -1
    bins2[~valid] = -1

    return bins1, bins2, edges1, edges2

---
## 5. Plotting Functions

In [ ]:
def plot_bivariate_legend(color_grid, edges1, edges2,
                          label1='Variable 1', label2='Variable 2',
                          figsize=(4, 4)):
    """Plot the 2D bivariate colorbar / legend.

    Parameters
    ----------
    color_grid : np.ndarray, shape (N, N, 3)
        From make_bivariate_colormap().
    edges1, edges2 : np.ndarray
        Bin edges for axis labels.
    label1, label2 : str
        Axis labels.
    figsize : tuple
        Figure size.

    Returns
    -------
    fig, ax
    """
    n = color_grid.shape[0]
    fig, ax = plt.subplots(figsize=figsize)

    # Draw colored squares
    for i in range(n):       # var2 (y-axis)
        for j in range(n):   # var1 (x-axis)
            rect = mpatches.FancyBboxPatch(
                (j, i), 1, 1,
                boxstyle='round,pad=0.02',
                facecolor=color_grid[i, j],
                edgecolor='white', linewidth=1.5,
            )
            ax.add_patch(rect)

    ax.set_xlim(0, n)
    ax.set_ylim(0, n)
    ax.set_aspect('equal')

    # Tick labels: show bin edge values at boundaries
    def _fmt(v):
        """Smart formatting for tick labels."""
        if abs(v) < 0.01 or abs(v) >= 1e4:
            return f"{v:.1e}"
        return f"{v:.3g}"

    ax.set_xticks(np.arange(n + 1))
    ax.set_xticklabels([_fmt(e) for e in edges1], fontsize=7, rotation=45, ha='right')
    ax.set_yticks(np.arange(n + 1))
    ax.set_yticklabels([_fmt(e) for e in edges2], fontsize=7)

    ax.set_xlabel(f"{label1}  →", fontsize=10, fontweight='bold')
    ax.set_ylabel(f"{label2}  →", fontsize=10, fontweight='bold')
    ax.set_title('Bivariate Legend', fontsize=11)

    # Add arrows in bottom-left corner
    ax.annotate('', xy=(n * 0.4, -0.15 * n), xytext=(0, -0.15 * n),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
                annotation_clip=False)
    ax.annotate('', xy=(-0.15 * n, n * 0.4), xytext=(-0.15 * n, 0),
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5),
                annotation_clip=False)

    fig.tight_layout()
    return fig, ax


def plot_bivariate_front_map(
    df, label1, label2, bins1, bins2, color_grid,
    lat_col='centroid_lat', lon_col='centroid_lon',
    projection='Robinson',
    marker_size=1.0,
    use_spatial_binning=False,
    spatial_bin_deg=2,
    title=None,
    figsize=(18, 9),
):
    """Plot fronts on a global map colored by bivariate bin.

    Parameters
    ----------
    df : pd.DataFrame
        Enriched front table with lat/lon columns.
    label1, label2 : str
        Display labels for the two variables (for title annotation).
    bins1, bins2 : np.ndarray of int
        Bin indices from assign_bivariate_bins().
    color_grid : np.ndarray, shape (N, N, 3)
        From make_bivariate_colormap().
    lat_col, lon_col : str
        Column names for front latitude/longitude.
    projection : str
        Cartopy projection name.
    marker_size : float
        Scatter marker size.
    use_spatial_binning : bool
        If True, aggregate into spatial bins (dominant category per cell).
    spatial_bin_deg : float
        Size of spatial bins in degrees.
    title : str, optional
        Custom title. Auto-generated if None.
    figsize : tuple
        Figure size.

    Returns
    -------
    fig, ax
    """
    # Get projection object
    proj_map = {
        'Robinson': ccrs.Robinson(),
        'Mollweide': ccrs.Mollweide(),
        'PlateCarree': ccrs.PlateCarree(),
    }
    proj = proj_map.get(projection, ccrs.Robinson())
    tfm = ccrs.PlateCarree()

    fig, ax = plt.subplots(figsize=figsize, subplot_kw={'projection': proj})
    ax.set_global()

    # Filter to valid fronts (both bins assigned)
    valid = (bins1 >= 0) & (bins2 >= 0)
    lats = df[lat_col].values[valid]
    lons = df[lon_col].values[valid]
    b1 = bins1[valid]
    b2 = bins2[valid]

    # Assign a color to each front based on its (bin1, bin2) pair
    front_colors = color_grid[b2, b1]  # index: [var2_bin, var1_bin]

    if use_spatial_binning:
        # ── Spatially bin fronts: assign the *most common* bivariate
        #    category to each spatial cell ──────────────────────────
        n_bv = color_grid.shape[0]
        biv_idx = b2 * n_bv + b1  # flatten 2D bin to 1D index

        n_lat = int(180 / spatial_bin_deg)
        n_lon = int(360 / spatial_bin_deg)
        lat_edges = np.linspace(-90, 90, n_lat + 1)
        lon_edges = np.linspace(-180, 180, n_lon + 1)

        # Use binned_statistic_2d just to get bin assignments
        from scipy.stats import binned_statistic_2d
        stat_result = binned_statistic_2d(
            lats, lons, biv_idx.astype(float),
            statistic='mean',
            bins=[lat_edges, lon_edges],
            expand_binnumbers=True,
        )

        # Count per spatial bin to get mode (most common category)
        n_total_bv = n_bv * n_bv
        count_grid = np.zeros((n_lat, n_lon), dtype=int)

        lat_bin_idx = np.clip(stat_result.binnumber[0] - 1, 0, n_lat - 1)
        lon_bin_idx = np.clip(stat_result.binnumber[1] - 1, 0, n_lon - 1)

        for k in range(len(lats)):
            li, lj = lat_bin_idx[k], lon_bin_idx[k]
            count_grid[li, lj] += 1

        # For each spatial bin, find the most common bivariate category
        cat_counts = np.zeros((n_lat, n_lon, n_total_bv), dtype=int)
        for k in range(len(lats)):
            li, lj = lat_bin_idx[k], lon_bin_idx[k]
            cat_counts[li, lj, int(biv_idx[k])] += 1

        mode_idx = np.argmax(cat_counts, axis=2)

        # Build an RGBA image
        has_data = count_grid > 0
        rgba_grid = np.ones((n_lat, n_lon, 4))
        rgba_grid[:, :, 3] = 0.0  # transparent by default

        for i in range(n_lat):
            for j in range(n_lon):
                if has_data[i, j]:
                    cat = mode_idx[i, j]
                    bi = cat % n_bv
                    bj = cat // n_bv
                    rgba_grid[i, j, :3] = color_grid[bj, bi]
                    rgba_grid[i, j, 3] = 1.0

        # Plot cell-by-cell as colored rectangles
        for i in range(n_lat):
            for j in range(n_lon):
                if has_data[i, j]:
                    ax.fill_between(
                        [lon_edges[j], lon_edges[j+1]],
                        lat_edges[i], lat_edges[i+1],
                        color=rgba_grid[i, j, :3],
                        transform=tfm, zorder=1,
                    )

        n_plotted = has_data.sum()
        method_str = f'{spatial_bin_deg}°×{spatial_bin_deg}° bins, mode category'

    else:
        # ── Scatter each front centroid directly ─────────────────────
        ax.scatter(
            lons, lats,
            c=front_colors,
            s=marker_size,
            transform=tfm,
            zorder=1,
            rasterized=True,
            edgecolors='none',
        )
        n_plotted = valid.sum()
        method_str = 'front centroids'

    # ── Map features (matching Turner_Angle_Global_Viz.ipynb style) ────────
    ax.add_feature(cfeature.LAND,      facecolor='lightgray', zorder=2)
    ax.add_feature(cfeature.COASTLINE, linewidth=0.4, color='k',    zorder=3)
    ax.add_feature(cfeature.BORDERS,   linewidth=0.2, color='gray', zorder=3)
    ax.gridlines(draw_labels=False, linewidth=0.3, color='gray',
                 alpha=0.5, linestyle='--')

    if title is None:
        title = (
            f'Bivariate Front Map: {label1} × {label2}\n'
            f'({n_plotted:,} {method_str})'
        )
    ax.set_title(title, fontsize=12, pad=10)

    fig.tight_layout()
    return fig, ax

---
## 6. Generate Bivariate Map

In [ ]:
# ─── COMPUTE BIVARIATE BINS ───────────────────────────────────────────────────
bins1, bins2, edges1, edges2 = assign_bivariate_bins(
    values1, values2, n_bins=N_BINS, clip_pct=CLIP_PERCENTILE
)

# Build the N×N color grid
color_grid = make_bivariate_colormap(N_BINS)

# Report bin assignments
valid_mask = (bins1 >= 0) & (bins2 >= 0)
print(f"Valid fronts (both variables finite): {valid_mask.sum():,} / {len(bins1):,}")
print(f"\ndiv/|f| bin edges: {edges1}")
print(f"Ro bin edges:      {edges2}")

# Show distribution across bivariate bins
print(f"\nBivariate bin counts ({N_BINS}×{N_BINS}):")
print(f"  rows = Ro bins (0=low … {N_BINS-1}=high)")
print(f"  cols = div/|f| bins (0=low … {N_BINS-1}=high)")
for i in range(N_BINS):
    row = []
    for j in range(N_BINS):
        count = ((bins1 == j) & (bins2 == i)).sum()
        row.append(f"{count:6d}")
    print(f"  Ro bin {i}: " + " | ".join(row))

In [ ]:
# ─── PLOT 1: BIVARIATE LEGEND ─────────────────────────────────────────────────
fig_legend, ax_legend = plot_bivariate_legend(
    color_grid, edges1, edges2,
    label1=VAR1_LABEL, label2=VAR2_LABEL,
    figsize=(4.5, 4.5),
)
plt.show()

In [ ]:
# ─── PLOT 2: BIVARIATE GLOBAL MAP ─────────────────────────────────────────────
fig_map, ax_map = plot_bivariate_front_map(
    df_enriched, VAR1_LABEL, VAR2_LABEL,
    bins1, bins2, color_grid,
    projection=PROJECTION,
    marker_size=MARKER_SIZE,
    use_spatial_binning=USE_SPATIAL_BINNING,
    spatial_bin_deg=SPATIAL_BIN_DEG,
)
plt.show()

In [ ]:
# ─── SAVE FIGURES ─────────────────────────────────────────────────────────────
SAVE_DIR.mkdir(parents=True, exist_ok=True)

tag = f"div_over_absf_x_rossby_{STATISTIC}_{N_BINS}bins"

legend_path = SAVE_DIR / f"bivariate_legend_{tag}.png"
fig_legend.savefig(legend_path, dpi=SAVE_DPI, bbox_inches='tight', facecolor='white')
print(f"Saved legend → {legend_path}")

map_path = SAVE_DIR / f"bivariate_map_{tag}.png"
fig_map.savefig(map_path, dpi=SAVE_DPI, bbox_inches='tight', facecolor='white')
print(f"Saved map    → {map_path}")

---
## 7. Quick Experiments

Cells below let you quickly re-run with different settings without
scrolling back to the config cell.

In [ ]:
# ─── TRY SCATTER MODE (no spatial binning) ───────────────────────────────────
fig_sc, ax_sc = plot_bivariate_front_map(
    df_enriched, VAR1_LABEL, VAR2_LABEL,
    bins1, bins2, color_grid,
    projection=PROJECTION,
    marker_size=0.3,
    use_spatial_binning=False,
    title=f'Bivariate Front Map (scatter): {VAR1_LABEL} × {VAR2_LABEL}',
)
plt.show()

scatter_path = SAVE_DIR / f"bivariate_scatter_{tag}.png"
fig_sc.savefig(scatter_path, dpi=SAVE_DPI, bbox_inches='tight', facecolor='white')
print(f"Saved scatter map → {scatter_path}")

In [ ]:
# ─── TRY N_BINS=4 ─────────────────────────────────────────────────────────────
N_BINS_ALT = 4

bins1_4, bins2_4, edges1_4, edges2_4 = assign_bivariate_bins(
    values1, values2, n_bins=N_BINS_ALT, clip_pct=CLIP_PERCENTILE
)
color_grid_4 = make_bivariate_colormap(N_BINS_ALT)

fig_leg4, _ = plot_bivariate_legend(
    color_grid_4, edges1_4, edges2_4,
    label1=VAR1_LABEL, label2=VAR2_LABEL, figsize=(5, 5),
)
plt.show()

fig_map4, _ = plot_bivariate_front_map(
    df_enriched, VAR1_LABEL, VAR2_LABEL,
    bins1_4, bins2_4, color_grid_4,
    projection=PROJECTION,
    use_spatial_binning=USE_SPATIAL_BINNING,
    spatial_bin_deg=SPATIAL_BIN_DEG,
)
plt.show()